## Visualization 2

In [2]:
## Scientific calculus
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

## Plots
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import plotly.graph_objects as go
from matplotlib.transforms import Bbox, TransformedBbox
from matplotlib.image import BboxImage
from matplotlib.patches import Circle

## Text, OS and images
from adjustText import adjust_text
import matplotlib.patheffects as path_effects
from matplotlib.legend_handler import HandlerBase
import matplotlib.font_manager as fm
import matplotlib.image as mpimg
import os
import base64
from PIL import Image

### Import Dataset and recover DFs

In [3]:
## Load Pokemon fonts
try:
    retro_font = fm.FontProperties(fname='../FONTS/PressStart2P-Regular.ttf')
    firered_font = fm.FontProperties(fname='../FONTS/pokemon_fire_red.ttf')
    emerald_font = fm.FontProperties(fname='../FONTS/pokemon-emerald.otf')
except FileNotFoundError:
    retro_font = fm.FontProperties(family='monospace', weight='bold')
    firered_font = fm.FontProperties(family='monospace', weight='bold')
    emerald_font = fm.FontProperties(family='monospace', weight='bold')

poke_font = '"Pokemon Emerald", Arial'

# Load the dataset into a pandas DataFrame
file_name = '../DATASETS/pokemon_complete_2025.csv'
POKE_df = pd.read_csv(file_name)

symbol_folder="../IMAGES/type_symbols"
icon_folder="../IMAGES/type_icons"

# Helper function to bypass browser security in case of browser rendering
def get_base64_image(image_path):
    with open(image_path, "rb") as img_file:
        b64_string = base64.b64encode(img_file.read()).decode('utf-8')
    return f"data:image/png;base64,{b64_string}"

medians_df = POKE_df.groupby('type_1')[['physical_total', 'special_total']].median().reset_index()

# Calculate perfect 1:1 bounding box
min_val = min(medians_df['physical_total'].min(), medians_df['special_total'].min()) - 50
max_val = max(medians_df['physical_total'].max(), medians_df['special_total'].max()) + 50


### Building Figures

In [4]:
fig = go.Figure()

## ............................................ ##
## Invisible Scatter (For Interactive Tooltips) ##
## ............................................ ##
## We use invisible dots to still get hover effects when mouseing over the images

fig.add_trace(go.Scatter(
    x=medians_df['physical_total'],
    y=medians_df['special_total'],
    mode='markers',
    marker=dict(size=35, color='rgba(0,0,0,0)'),
    customdata=medians_df['type_1'],
    hovertemplate=(
        "<b>TYPE: %{customdata}</b><br>"
        "MEDIAN PHYSICAL: %{x}<br>"
        "MEDIAN SPECIAL: %{y}"
        "<extra></extra>" 
    ),
    showlegend=False 
))

  
## INJECTING DATA IMAGES ##
for idx, row in medians_df.iterrows():
    type_name = row['type_1']
    x = row['physical_total']
    y = row['special_total']
    
    img_path = f"{symbol_folder}/{type_name}.png"
    
    b64_img = get_base64_image(img_path)
    fig.add_layout_image(
        dict(
            source=b64_img,
            xref="x", yref="y",
            x=x, y=y,
            sizex=8, sizey=8,
            xanchor="center", yanchor="middle",
            layer="above"
        )
    )


## ................. ##
## Add Diagonal line ##
## ................. ##
fig.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="#404040", width=2, dash="dash"),
    opacity=0.5, 
    layer="below"
)

## .................. ##
## Add Shadowed areas ##
## .................. ##

## About Physical types
fig.add_shape(
    type="path",
    xref="x", yref="y",
    path="M 181 143 C 200 155, 213 145, 213 140 C 213 130, 200 125, 185 110 C 170 95, 165 130, 181 143 Z",
    fillcolor="#D68548",
    line=dict(color="rgba(0,0,0,0)", width=2, dash="dot"),
    opacity=0.2, 
    layer="below"
)

## About Special types
fig.add_shape(
    type="path",
    xref="x", yref="y",
    path="M 122 190 C 130 190, 148 185, 134 155 C 135 155, 130 149, 128 150 C 125 149, 108 168, 119 188 Z",
    fillcolor="#A36ECC",           
    line=dict(color="rgba(0,0,0,0)", width=2, dash="dot"), # transparent border
    opacity=0.2, 
    layer="below"
)

## Add their correspondent annotation
fig.add_annotation(
    xref="x", yref="y",
    x=115, y=175,           # Positioned to the left of the purple blob
    text="<span style='color:#A36ECC; font-size:18px'><b>ETHEREAL FORCES</b></span><br><span style='font-size:14px'>Psychic, Fairy & Ghost</span><br><span style='font-size:12px'><i>Harnessing otherworldly power<br>that transcends physical limits.</i></span>",
    showarrow=False, 
    xanchor="right",        # Anchors the right side of the text to the X coordinate
    yanchor="middle",
    font=dict(family=poke_font, color="#5A5665"),
    align="right"           # Right-aligns the text block itself
)

fig.add_annotation(
    xref="x", yref="y",
    x=195, y=113,           # Positioned to the right of the orange blob
    text="<span style='color:#D68548; font-size:18px'><b>BRUTE FORCE</b></span><br><span style='font-size:14px'>Ground, Fighting, Rock & Steel</span><br><span style='font-size:12px'><i>Built for close combat and<br>heavy physical impacts.</i></span>",
    showarrow=False, 
    xanchor="left",         # Anchors the left side of the text to the X coordinate
    yanchor="middle",
    font=dict(family=poke_font, color="#5A5665"),
    align="left"            # Left-aligns the text block itself
)

## ................ ##
## Legend and boxes ##
## ................ ##
legend_types = medians_df['type_1'].unique()
num_items = len(legend_types)

## Params of the legend 
legend_x_start = 1.04      # Distance of the whole legend from the right edge of the plot
legend_width = 0.19        # Size (width) of the legend box
symbol_x = 1.09            # X position of the circular symbols
icon_x = 1.12              # X position of the type name image
box_top = 1.01             # Anchor the top of the box exactly to the plot's upper edge (1.0)
title_y = box_top - 0.03   # Y position of the Title
start_y = title_y - 0.08   # Y position of the FIRST item in the list
step_y = 0.047             # Vertical distance between items in the list
box_height = (num_items * step_y) + 0.12 
box_bottom = box_top - box_height

## Box outer
fig.add_shape(
    type="rect", xref="paper", yref="paper",
    x0=legend_x_start, 
    y0=box_bottom, 
    x1=legend_x_start + legend_width, 
    y1=box_top, 
    fillcolor="#41455a", line_width=2, layer="below"
)

## Inner box
fig.add_shape(
    type="rect", xref="paper", yref="paper",
    x0=legend_x_start + 0.005, 
    y0=box_bottom + 0.005, 
    x1=legend_x_start + legend_width - 0.005, 
    y1=box_top - 0.005,
    fillcolor="#f8f8f8", line=dict(color="#6d687d", width=4.5), layer="below"
)

## Adding the title
fig.add_annotation(
    xref="paper", yref="paper",
    x=legend_x_start + 0.018, # Title sits slightly indented from the left border
    y=title_y,
    text="<b>TYPES:</b>",
    showarrow=False, xanchor="left",
    font=dict(size=20, family=poke_font, color="#4f504e")
)

## Use type simbols inside the legend
for i, type_name in enumerate(legend_types):
    curr_y = start_y - (i * step_y)

    # images paths
    symbol_path = f"{symbol_folder}/{type_name}.png"
    icon_path = f"{icon_folder}/{type_name}.png"

    # Circular symbols
    if os.path.exists(symbol_path):
        img_sym = Image.open(symbol_path)
        fig.add_layout_image(
            dict(
                source=img_sym,
                xref="paper", yref="paper", 
                x=symbol_x,             # Anchored to symbol_x
                y=curr_y,
                sizex=0.045, sizey=0.045,   
                xanchor="center", yanchor="middle",
                layer="above"
            )
        )
    
    # Type pill icons
    if os.path.exists(icon_path):
        img_icon = Image.open(icon_path)
        
        fig.add_layout_image(
            dict(
                source=img_icon,
                xref="paper", yref="paper", 
                x=icon_x,               # Anchored to icon_x
                y=curr_y,
                sizex=0.1, sizey=0.035,
                xanchor="left", yanchor="middle",
                layer="above"
            )
        )


## ............ ##
## Figure SETUP ##
## ............ ##

fig.update_layout(
    
    ## Title and general setup
    title=dict(
        text="<b>POKéMON TYPES STAY TRUE TO THEIR NATURE</b><br><sup>Comparing the <b>median</b> total stats (Attack + Defense) across all types</sup>",
        font=dict(family=poke_font, size=32, color="#5A5665"),
        pad=dict(b=35),
        x=0.07,
        # x=0.11,
        xanchor="left",
        xref="container",
        y=0.83
    ),
    margin=dict(l=125, r=270, t=210, b=180),
    height=820,  
    width=1090,
 
    
    plot_bgcolor="#fcfcfc",
    paper_bgcolor="rgba(0,0,0,0)",

    ## XAXIS
    xaxis=dict(
        title=dict(
            text="<b>PHYSICAL STATS</b>", 
            font=dict(size=22, family=poke_font, color="#D64848")
        ),
        tickfont=dict(family=poke_font, size=22, color="#5A5665"),
        range=[70, 240],
        dtick=20,
        showgrid=True, 
        gridwidth=2,
        gridcolor="rgba(176, 184, 192, 0.2)",
        griddash="2px, 4px",
        showline=True,
        linecolor="#6d687d",          
        linewidth=4.5,
        mirror=True,
        zeroline=False
    ),

    ## YAXIS
    yaxis=dict(
        title=dict(
            text="<b>SPECIAL STATS</b>", 
            font=dict(size=22, family=poke_font, color="#D64848")
        ),
        tickfont=dict(family=poke_font, size=22, color="#5A5665"),
        scaleanchor="x",    # Locks the X-axis scale to the Y-axis
        scaleratio=1, 
        range=[110, 190],
        dtick=20,
        visible=True, 
        showgrid=True,
        gridwidth=4, 
        gridcolor="rgba(176, 184, 192, 0.2)", 
        griddash="2px, 4px",
        ticks="outside",
        ticklen=20,
        showline=True,
        linecolor="#6d687d",          
        linewidth=4.5,
        mirror=True,
        zeroline=False
    ),
)

## ....... ##
## Caption ##
## ....... ##



fig.add_annotation(
    xref="paper", yref="paper",
    x=-0.07, y=-0.275,
    text="<b>FIGURE</b>.  Comparison of median total stats (attack + defense) across all 18 POKéMON types up to Gen IX, where the dashed line ",
    showarrow=False,
    xanchor="left", yanchor="bottom", 
    align="left",   
    font=dict(family=poke_font, size=19, color="#5A5665")
)

fig.add_annotation(
    xref="paper", yref="paper",
    x=-0.004, y=-0.265,
    text="   represents a balanced 1:1 statistics ratio. The highlighted regions do emphasize the most heavily skewed types. ",
    showarrow=False,
    xanchor="left", yanchor="top",    
    align="left",                     
    font=dict(family=poke_font, size=19, color="#5A5665")
)

fig.add_annotation(
    xref="paper", yref="paper",
    x=-0.005, y=-0.31,
    text="   Source -  https://www.kaggle.com/datasets/darkmatternet/ultimate-pokmon-dataset-2025",
    showarrow=False,
    xanchor="left", yanchor="top",    
    align="left",                     
    font=dict(family=poke_font, size=19, color="#5A5665")
)


def rounded_rect(x0, y0, x1, y1, rx, ry):
    return (
        f"M {x0+rx},{y0} "
        f"L {x1-rx},{y0} "
        f"Q {x1},{y0} {x1},{y0+ry} "
        f"L {x1},{y1-ry} "
        f"Q {x1},{y1} {x1-rx},{y1} "
        f"L {x0+rx},{y1} "
        f"Q {x0},{y1} {x0},{y1-ry} "
        f"L {x0},{y0+ry} "
        f"Q {x0},{y0} {x0+rx},{y0} Z"
    )

cap_x0, cap_x1 = -0.1, 1.24 
cap_y0, cap_y1 = -0.38, -0.20 # Queste rimangono invariate

dx_black, dy_black = 0.002, 0.004
dx_red, dy_red = 0.015, 0.008
rx_out = 0.015
ry_out = 0.035

fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(cap_x0, cap_y0, cap_x1, cap_y1, rx_out, ry_out),
    fillcolor="#000000", line_width=0, layer="below"
)


fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(
        cap_x0 + dx_black, cap_y0 + dy_black, 
        cap_x1 - dx_black, cap_y1 - dy_black, 
        rx_out * 0.9, ry_out * 0.9
    ),
    fillcolor="#D64848", line_width=0, layer="below"
)

fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(
        cap_x0 + dx_red, cap_y0 + dy_red, 
        cap_x1 - dx_red, cap_y1 - dy_red, 
        rx_out * 0.6, ry_out * 0.6
    ),
    fillcolor="#F8F8F8", line_width=0, layer="below"
)


## Download configuration
config_download = {
  'toImageButtonOptions': {
    'format': 'png', 
    'filename': 'Vis_2_highres',
    'height': 850,  
    'width':1090,    
    'scale': 4           
  }
}
fig.show(config=config_download)

In [24]:
fig = go.Figure()

## ............................................ ##
## Invisible Scatter (For Interactive Tooltips) ##
## ............................................ ##
## We use invisible dots to still get hover effects when mouseing over the images

fig.add_trace(go.Scatter(
    x=medians_df['physical_total'],
    y=medians_df['special_total'],
    mode='markers',
    marker=dict(size=35, color='rgba(0,0,0,0)'),
    customdata=medians_df['type_1'],
    hovertemplate=(
        "<b>TYPE: %{customdata}</b><br>"
        "MEDIAN PHYSICAL: %{x}<br>"
        "MEDIAN SPECIAL: %{y}"
        "<extra></extra>" 
    ),
    showlegend=False 
))

  
## INJECTING DATA IMAGES ##
for idx, row in medians_df.iterrows():
    type_name = row['type_1']
    x = row['physical_total']
    y = row['special_total']
    
    img_path = f"{symbol_folder}/{type_name}.png"
    
    b64_img = get_base64_image(img_path)
    fig.add_layout_image(
        dict(
            source=b64_img,
            xref="x", yref="y",
            x=x, y=y,
            sizex=8, sizey=8,
            xanchor="center", yanchor="middle",
            layer="above"
        )
    )


## ................. ##
## Add Diagonal line ##
## ................. ##
fig.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="#404040", width=2, dash="dash"),
    opacity=0.5, 
    layer="below"
)

## .................. ##
## Add Shadowed areas ##
## .................. ##

## About Physical types
fig.add_shape(
    type="path",
    xref="x", yref="y",
    path="M 181 143 C 200 155, 213 145, 213 140 C 213 130, 200 125, 185 110 C 170 95, 165 130, 181 143 Z",
    fillcolor="#D68548",
    line=dict(color="rgba(0,0,0,0)", width=2, dash="dot"),
    opacity=0.2, 
    layer="below"
)

## About Special types
fig.add_shape(
    type="path",
    xref="x", yref="y",
    path="M 122 190 C 130 190, 148 185, 134 155 C 135 155, 130 149, 128 150 C 125 149, 108 168, 119 188 Z",
    fillcolor="#A36ECC",           
    line=dict(color="rgba(0,0,0,0)", width=2, dash="dot"), # transparent border
    opacity=0.2, 
    layer="below"
)

## Add their correspondent annotation
fig.add_annotation(
    xref="x", yref="y",
    x=115, y=175,           # Positioned to the left of the purple blob
    text="<span style='color:#A36ECC; font-size:18px'><b>ETHEREAL FORCES</b></span><br><span style='font-size:14px'>Psychic, Fairy & Ghost</span><br><span style='font-size:12px'><i>Harnessing otherworldly power<br>that transcends physical limits.</i></span>",
    showarrow=False, 
    xanchor="right",        # Anchors the right side of the text to the X coordinate
    yanchor="middle",
    font=dict(family=poke_font, color="#5A5665"),
    align="right"           # Right-aligns the text block itself
)

fig.add_annotation(
    xref="x", yref="y",
    x=195, y=113,           # Positioned to the right of the orange blob
    text="<span style='color:#D68548; font-size:18px'><b>BRUTE FORCE</b></span><br><span style='font-size:14px'>Ground, Fighting, Rock & Steel</span><br><span style='font-size:12px'><i>Built for close combat and<br>heavy physical impacts.</i></span>",
    showarrow=False, 
    xanchor="left",         # Anchors the left side of the text to the X coordinate
    yanchor="middle",
    font=dict(family=poke_font, color="#5A5665"),
    align="left"            # Left-aligns the text block itself
)

## ................ ##
## Legend and boxes ##
## ................ ##
legend_types = medians_df['type_1'].unique()
num_items = len(legend_types)

## Params of the legend 
legend_x_start = 1.04      # Distance of the whole legend from the right edge of the plot
legend_width = 0.19        # Size (width) of the legend box
symbol_x = 1.09            # X position of the circular symbols
icon_x = 1.12              # X position of the type name image
box_top = 1.01             # Anchor the top of the box exactly to the plot's upper edge (1.0)
title_y = box_top - 0.03   # Y position of the Title
start_y = title_y - 0.08   # Y position of the FIRST item in the list
step_y = 0.047             # Vertical distance between items in the list
box_height = (num_items * step_y) + 0.12 
box_bottom = box_top - box_height

## Box outer
fig.add_shape(
    type="rect", xref="paper", yref="paper",
    x0=legend_x_start, 
    y0=box_bottom, 
    x1=legend_x_start + legend_width, 
    y1=box_top, 
    fillcolor="#41455a", line_width=2, layer="below"
)

## Inner box
fig.add_shape(
    type="rect", xref="paper", yref="paper",
    x0=legend_x_start + 0.005, 
    y0=box_bottom + 0.005, 
    x1=legend_x_start + legend_width - 0.005, 
    y1=box_top - 0.005,
    fillcolor="#f8f8f8", line=dict(color="#6d687d", width=4.5), layer="below"
)

## Adding the title
fig.add_annotation(
    xref="paper", yref="paper",
    x=legend_x_start + 0.018, # Title sits slightly indented from the left border
    y=title_y,
    text="<b>TYPES:</b>",
    showarrow=False, xanchor="left",
    font=dict(size=20, family=poke_font, color="#4f504e")
)

## Use type simbols inside the legend
for i, type_name in enumerate(legend_types):
    curr_y = start_y - (i * step_y)

    # images paths
    symbol_path = f"{symbol_folder}/{type_name}.png"
    icon_path = f"{icon_folder}/{type_name}.png"

    # Circular symbols
    if os.path.exists(symbol_path):
        img_sym = Image.open(symbol_path)
        fig.add_layout_image(
            dict(
                source=img_sym,
                xref="paper", yref="paper", 
                x=symbol_x,             # Anchored to symbol_x
                y=curr_y,
                sizex=0.045, sizey=0.045,   
                xanchor="center", yanchor="middle",
                layer="above"
            )
        )
    
    # Type pill icons
    if os.path.exists(icon_path):
        img_icon = Image.open(icon_path)
        
        fig.add_layout_image(
            dict(
                source=img_icon,
                xref="paper", yref="paper", 
                x=icon_x,               # Anchored to icon_x
                y=curr_y,
                sizex=0.1, sizey=0.035,
                xanchor="left", yanchor="middle",
                layer="above"
            )
        )


## ............ ##
## Figure SETUP ##
## ............ ##

fig.update_layout(
    
    ## Title and general setup
    title=dict(
        text="<b>POKéMON TYPES STAY TRUE TO THEIR NATURE</b><br><sup>Comparing the <b>median</b> total stats (Attack + Defense) across all types</sup>",
        font=dict(family=poke_font, size=32, color="#5A5665"),
        pad=dict(b=35),
        x=0.07,
        # x=0.11,
        xanchor="left",
        xref="container",
        y=0.83
    ),
    margin=dict(l=125, r=270, t=210, b=180),
    height=820,  
    width=1090,
 
    
    plot_bgcolor="#fcfcfc",
    paper_bgcolor="rgba(0,0,0,0)",

    ## XAXIS
    xaxis=dict(
        title=dict(
            text="<b>PHYSICAL STATS</b>", 
            font=dict(size=22, family=poke_font, color="#D64848")
        ),
        tickfont=dict(family=poke_font, size=22, color="#5A5665"),
        range=[70, 240],
        dtick=20,
        showgrid=True, 
        gridwidth=2,
        gridcolor="rgba(176, 184, 192, 0.2)",
        griddash="2px, 4px",
        showline=True,
        linecolor="#6d687d",          
        linewidth=4.5,
        mirror=True,
        zeroline=False
    ),

    ## YAXIS
    yaxis=dict(
        title=dict(
            text="<b>SPECIAL STATS</b>", 
            font=dict(size=22, family=poke_font, color="#D64848")
        ),
        tickfont=dict(family=poke_font, size=22, color="#5A5665"),
        scaleanchor="x",    # Locks the X-axis scale to the Y-axis
        scaleratio=1, 
        range=[110, 190],
        dtick=20,
        visible=True, 
        showgrid=True,
        gridwidth=4, 
        gridcolor="rgba(176, 184, 192, 0.2)", 
        griddash="2px, 4px",
        ticks="outside",
        ticklen=20,
        showline=True,
        linecolor="#6d687d",          
        linewidth=4.5,
        mirror=True,
        zeroline=False
    ),
)

## ....... ##
## Caption ##
## ....... ##



fig.add_annotation(
    xref="paper", yref="paper",
    x=-0.07, y=-0.275,
    text="<b>FIGURE</b>.  Comparison of median total stats (att+def) across all 18 POKéMON types up to Gen IX, with dashed line representing a balanced 1:1 stats",
    showarrow=False,
    xanchor="left", yanchor="bottom", 
    align="left",   
    font=dict(family=poke_font, size=19, color="#5A5665")
)

fig.add_annotation(
    xref="paper", yref="paper",
    x=-0.004, y=-0.265,
    text="     ratio. Highlighted regions show the most heavily skewed types. [kaggle.com/datasets/darkmatternet/ultimate-pokmon-dataset-2025]",
    showarrow=False,
    xanchor="left", yanchor="top",    
    align="left",                     
    font=dict(family=poke_font, size=19, color="#5A5665")
)

fig.add_annotation(
    xref="paper", yref="paper",
    x=-0.005, y=-0.31,
    text="   ",
    showarrow=False,
    xanchor="left", yanchor="top",    
    align="left",                     
    font=dict(family=poke_font, size=19, color="#5A5665")
)


def rounded_rect(x0, y0, x1, y1, rx, ry):
    return (
        f"M {x0+rx},{y0} "
        f"L {x1-rx},{y0} "
        f"Q {x1},{y0} {x1},{y0+ry} "
        f"L {x1},{y1-ry} "
        f"Q {x1},{y1} {x1-rx},{y1} "
        f"L {x0+rx},{y1} "
        f"Q {x0},{y1} {x0},{y1-ry} "
        f"L {x0},{y0+ry} "
        f"Q {x0},{y0} {x0+rx},{y0} Z"
    )

cap_x0, cap_x1 = -0.1, 1.35
cap_y0, cap_y1 = -0.34, -0.20


dx_black, dy_black = 0.002, 0.004
dx_red, dy_red = 0.015, 0.008
rx_out = 0.015
ry_out = 0.035

fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(cap_x0, cap_y0, cap_x1, cap_y1, rx_out, ry_out),
    fillcolor="#000000", line_width=0, layer="below"
)


fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(
        cap_x0 + dx_black, cap_y0 + dy_black, 
        cap_x1 - dx_black, cap_y1 - dy_black, 
        rx_out * 0.9, ry_out * 0.9
    ),
    fillcolor="#D64848", line_width=0, layer="below"
)

fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(
        cap_x0 + dx_red, cap_y0 + dy_red, 
        cap_x1 - dx_red, cap_y1 - dy_red, 
        rx_out * 0.6, ry_out * 0.6
    ),
    fillcolor="#F8F8F8", line_width=0, layer="below"
)


## Download configuration
config_download = {
  'toImageButtonOptions': {
    'format': 'png', 
    'filename': 'Vis_2_highres',
    'height': 850,  
    'width':1090,    
    'scale': 4           
  }
}
fig.show(config=config_download)